In [ ]:
from bs4 import BeautifulSoup
import re

def extract_article_text(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    # Remove obvious noise
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside", "form"]):
        tag.decompose()

    # First try common article containers
    candidates = [
        soup.find("article"),
        soup.find("main"),
        soup.find(attrs={"role": "main"}),
        soup.find("div", class_=re.compile(r"article|content|post|story", re.I)),
    ]

    main_content = next((c for c in candidates if c is not None), soup)

    text = main_content.get_text(separator=" ", strip=True)
    text = re.sub(r"\s+", " ", text)

    return text

# Example
text = extract_article_text("/content/24_Source_1.html")
print(text[:3000])

FileNotFoundError: [Errno 2] No such file or directory: '/content/24_Source_1.html'

In [ ]:
import os
import re
from bs4 import BeautifulSoup

def extract_article_text(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    # ❌ Remove junk elements
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside", "form"]):
        tag.decompose()

    # 🎯 Try to find main article content
    candidates = [
        soup.find("article"),
        soup.find("main"),
        soup.find(attrs={"role": "main"}),
        soup.find("div", class_=re.compile(r"article|content|post|story|body", re.I)),
    ]

    main_content = next((c for c in candidates if c is not None), soup)

    # 🧹 Extract and clean text
    text = main_content.get_text(separator=" ", strip=True)
    text = re.sub(r"\s+", " ", text)

    return text


# =========================
# 📂 Process entire folder
# =========================
input_folder = "articles_html"
output_folder = "clean_text"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.endswith(".html"):
        file_path = os.path.join(input_folder, filename)

        try:
            clean_text = extract_article_text(file_path)

            # Save cleaned text
            output_file = os.path.join(output_folder, filename.replace(".html", ".txt"))
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(clean_text)

            print(f"✅ Processed: {filename}")

        except Exception as e:
            print(f"❌ Error with {filename}: {e}")

print("\n🎉 Done! Clean text saved in 'clean_text/' folder.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Rstudio.docx.gdoc', 'Colab Notebooks', 'roberta_model', 'NLP']


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "/content/drive/MyDrive/NLP/roberta_model"

model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

model.eval()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (

In [ ]:
import torch

text = "This campaign used misinformation to influence voters"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
)

with torch.no_grad():
    outputs = model(**inputs)

pred_id = torch.argmax(outputs.logits, dim=1).item()
pred_label = model.config.id2label[pred_id]

print("Prediction:", pred_label)

Prediction: Adversary Action


In [ ]:
# Install once
!pip install transformers datasets torch -q

import os
os.environ["WANDB_DISABLED"] = "true"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# =========================
# 1) Load dataset
# =========================
df2 = pd.read_csv("adversary_actions_dataset.csv", engine="python", on_bad_lines="skip")

df2["justification"] = df2["justification"].fillna("")
df2["text"] = df2["statement"] + " " + df2["justification"]

# =========================
# 2) Encode labels
# =========================
labels_unique = sorted(df2["label"].unique())
label2id = {label: i for i, label in enumerate(labels_unique)}
id2label = {i: label for label, i in label2id.items()}

df2["labels"] = df2["label"].map(label2id)

# =========================
# 3) Train/test split
# =========================
train_df, test_df = train_test_split(
    df2[["text", "labels", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=df2["label"]
)

# =========================
# 4) Convert to HuggingFace Dataset
# =========================
train_dataset = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)

# =========================
# 5) Tokenizer
# =========================
model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

# =========================
# 6) Load model
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# =========================
# 7) Metrics
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted")
    }

# =========================
# 8) Training setup
# =========================
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# =========================
# 9) Train
# =========================
trainer.train()

# =========================
# 10) Evaluate
# =========================
results = trainer.evaluate()
print("\nEvaluation results:")
print(results)

# =========================
# 11) Detailed classification report
# =========================
pred_output = trainer.predict(test_dataset)
y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

print("\nClassification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))]
))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/lo

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
!pip install transformers datasets torch scikit-learn pandas -q

import os
os.environ["WANDB_DISABLED"] = "true"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

drive.mount('/content/drive')

# =========================
# 1) Define paths
# =========================
DATA_PATH = "/content/countermeasures_dataset.csv"
SAVE_PATH = "/content/drive/MyDrive/NLP/countermeasures_roberta_model"

# =========================
# 2) Load dataset
# =========================
df = pd.read_csv(DATA_PATH, engine="python", on_bad_lines="skip")

df["justification"] = df["justification"].fillna("")
df["text"] = df["statement"] + " " + df["justification"]

# =========================
# 3) Encode labels
# =========================
labels_unique = sorted(df["label"].unique())
label2id = {label: i for i, label in enumerate(labels_unique)}
id2label = {i: label for label, i in label2id.items()}

df["labels"] = df["label"].map(label2id)

# =========================
# 4) Train/test split
# =========================
train_df, test_df = train_test_split(
    df[["text", "labels", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# =========================
# 5) Convert to HuggingFace Dataset
# =========================
train_dataset = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)

# =========================
# 6) Tokenizer
# =========================
model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

# =========================
# 7) Load model
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# =========================
# 8) Metrics
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted")
    }

# =========================
# 9) Training setup
# =========================
training_args = TrainingArguments(
    output_dir="./results_countermeasures",
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs_countermeasures"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# =========================
# 10) Train
# =========================
trainer.train()

# =========================
# 11) Evaluate
# =========================
results = trainer.evaluate()
print("\nEvaluation results:")
print(results)

# =========================
# 12) Detailed classification report
# =========================
pred_output = trainer.predict(test_dataset)
y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

print("\nClassification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))]
))

# =========================
# 13) Save model and tokenizer
# =========================
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"\nModel and tokenizer saved to: {SAVE_PATH}")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...


Evaluation results:
{'eval_loss': 0.12093120813369751, 'eval_accuracy': 0.9662921348314607, 'eval_f1_weighted': 0.965713526168524, 'eval_runtime': 79.7471, 'eval_samples_per_second': 2.232, 'eval_steps_per_second': 0.564, 'epoch': 1.0}


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Classification Report:

                              precision    recall  f1-score   support

      Applied Countermeasure       0.94      0.97      0.96        33
Countermeasure Effectiveness       0.96      0.86      0.91        28
  Non-Countermeasure Related       0.97      1.00      0.98        61
  Recommended Countermeasure       0.98      0.98      0.98        56

                    accuracy                           0.97       178
                   macro avg       0.96      0.95      0.96       178
                weighted avg       0.97      0.97      0.97       178



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model and tokenizer saved to: /content/drive/MyDrive/NLP/countermeasures_roberta_model


In [ ]:
trainer.save_model("roberta_model")
tokenizer.save_pretrained("roberta_model")